In [1]:
# 1. Setup Environment
!pip install transformers==4.46.2 tokenizers==0.20.3 sentencepiece protobuf bitsandbytes 
print("✅ Dependencies installed. Please RESTART SESSION if you see version conflicts.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 57.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 84.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
✅ Dependencies installed. Please RESTART SE

In [ ]:
import os
import sys
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    TrainerCallback
)
import numpy

# --- SENIOR DEV FIX: PyTorch 2.6+ Checkpoint Resuming ---
# PyTorch 2.6 broke HuggingFace checkpoint resuming by strictly enforcing weights_only=True.
# Playing whack-a-mole with safe_globals for every single numpy type is brittle.
# We safely monkey-patch torch.load to disable this strict check globally.
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load
# --------------------------------------------------------------

# ================== CONFIGURATION ==================
# Model and Data Paths
# Phase 2: Load weights from the highly successful 8550 checkpoint instead of base CodeT5
MODEL_NAME = "/kaggle/input/datasets/saffigaming/zeroshot02/model_checkpoints/checkpoint-9604"
# ⚠️ MAKE SURE YOU UPLOAD THE NEW ZERO-SHOT DATASET TO KAGGLE AND UPDATE THIS PATH:
DATA_FILE = "/kaggle/input/datasets/saffiullah892/zeroshot01/zeroshot_train_final.jsonl"
OUTPUT_DIR = "/kaggle/working/model_checkpoints"

# We set this to None because we want a FRESH learning rate for the new zero-shot dataset.
# If we "resume", the learning rate will be 0.0 (since the previous run finished) and the model won't learn anything!
RESUME_FROM_CHECKPOINT = None

# Hyperparameters - Memory Safe for Tesla T4
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 512       
BATCH_SIZE = 4                
GRAD_ACC = 4                  
EPOCHS = 18                   # Extended epochs to allow zero-shot learning to settle
LEARNING_RATE = 2e-5      
# ===================================================

def train():
    # 1. Setup Environment
    os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Check GPU availability
    num_gpus = torch.cuda.device_count()
    print(f"✅ Device: {device.upper()}")
    if device == "cuda":
        print(f"✅ Number of GPUs available: {num_gpus}")

    # 2. Load Tokenizer & Model
    print("\nLoading model and tokenizer...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
        
        # if hasattr(torch, 'compile') and device == "cuda":
        #     print("🚀 Compiling model with torch.compile...")
        #     model = torch.compile(model)
            
    except (ValueError, TypeError) as e:
        print(f"\n❌ Error loading tokenizer/model: {e}")
        print("\n" + "!"*50)
        print("💡 HINT: This error often occurs in Kaggle due to an outdated 'transformers' library.")
        print("💡 FIX: Run the following in a separate code cell at the TOP of your notebook:")
        print("\n   !pip install --upgrade transformers sentencepiece protobuf\n")
        print("Then restart the session and run the script again.")
        print("!"*50 + "\n")
        sys.exit(1)
    
    # Memory Optimizations
    # model.gradient_checkpointing_enable() # DISABLED for speed

    # 3. Load Dataset
    print(f"\nLoading dataset from {DATA_FILE}...")
    dataset = load_dataset("json", data_files=DATA_FILE, split="train")
    
    # Split for validation
    dataset = dataset.train_test_split(test_size=0.1, seed=42)
    train_ds = dataset["train"]
    eval_ds = dataset["test"]

    def preprocess(examples):
        model_inputs = tokenizer(
            examples["input"], 
            max_length=MAX_INPUT_LENGTH, 
            padding=False, 
            truncation=True
        )
        labels = tokenizer(
            text_target=examples["target"], 
            max_length=MAX_TARGET_LENGTH, 
            padding=False, 
            truncation=True
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    print("Tokenizing dataset...")
    train_ds = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
    eval_ds = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

    # 4. Training Arguments
    args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        logging_steps=50,                  # Log every 50 steps to see progress faster
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC,
        weight_decay=0.01,
        num_train_epochs=EPOCHS,
        predict_with_generate=True,
        fp16=True if torch.cuda.is_available() else False,
        
        # Checkpointing and Best Model Settings
        load_best_model_at_end=True,       # Load the best checkpoint at the end
        metric_for_best_model="loss",      # Define 'best' by lowest validation loss
        save_total_limit=2,                # Keeps only the Best and the Last checkpoint to save disk space
        
        # Progress Bar and Monitoring
        disable_tqdm=False,                
        report_to="none",                  
        gradient_checkpointing=False,      
        ddp_find_unused_parameters=False,   
        
        # Performance Optimizations
        optim="adamw_bnb_8bit",
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        group_by_length=True,
        
        # [MEMORY SAFE] Disabled for T4 to prevent OOM
        torch_compile=False,
        remove_unused_columns=False
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

    # 5. Trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=data_collator,
        processing_class=tokenizer
    )

    print("\n" + "="*50)
    print("🚀 STARTING SENIOR-OPTIMIZED TRAINING")
    print(f"Device: {device.upper()} | GPUs: {num_gpus} | Precision: FP16")
    print(f"Optimizer: 8-bit AdamW | Compiled: {'Yes' if args.torch_compile else 'No'}")
    print(f"Effective Batch Size: {BATCH_SIZE * GRAD_ACC * num_gpus}")
    print("="*50)

    # 6. Start Training
    try:
        if RESUME_FROM_CHECKPOINT and os.path.exists(RESUME_FROM_CHECKPOINT):
            print(f"🔄 Resuming from checkpoint: {RESUME_FROM_CHECKPOINT}")
            trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
        else:
            trainer.train()
    except Exception as e:
        print(f"\n❌ Training failed: {e}")
        if "bitsandbytes" in str(e).lower():
            print("💡 HINT: bitsandbytes installation might be incomplete. Run '!pip install bitsandbytes' and restart.")
    
    print(f"\n✅ Training Complete. Saving Best Model to {OUTPUT_DIR}")
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

if __name__ == "__main__":
    train()


2026-05-07 02:40:05.434965: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778121605.621925      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778121605.684497      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778121606.137992      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778121606.138026      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778121606.138029      57 computation_placer.cc:177] computation placer alr

✅ Device: CUDA
✅ Number of GPUs available: 2

Loading model and tokenizer...

Loading dataset from /kaggle/input/datasets/saffiullah892/zeroshot01/zeroshot_train_final.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

Tokenizing dataset...


Map:   0%|          | 0/10987 [00:00<?, ? examples/s]

Map:   0%|          | 0/1221 [00:00<?, ? examples/s]


🚀 STARTING SENIOR-OPTIMIZED TRAINING
Device: CUDA | GPUs: 2 | Precision: FP16
Optimizer: 8-bit AdamW | Compiled: No
Effective Batch Size: 32


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Step,Training Loss,Validation Loss
